# Project Details
Fill in your details in the cell below before running the rest of the notebook.


In [ ]:
# ==== EDIT THESE BEFORE RUNNING ====
STUDENT_NAME = "Abhay"
ROLL_NUMBER = ""        # e.g. "CBS2023045"
COURSE_SECTION = ""     # e.g. "BBA-AI, Section B"
COLLEGE = "Chitkara Business School"
PROJECT_TITLE = "AI in Digital Wealth Management: Scripbox Robo-Advisory Case Study"
# ====================================

print(f"Project: {PROJECT_TITLE}")
print(f"Student: {STUDENT_NAME}")
print(f"Roll No: {ROLL_NUMBER if ROLL_NUMBER else '[fill in above]'}")
print(f"Course/Section: {COURSE_SECTION if COURSE_SECTION else '[fill in above]'}")
print(f"College: {COLLEGE}")


# Scripbox Robo-Advisory — AI/ML Model Demo
### Business Case Study: AI in Digital Wealth Management (India)

This notebook demonstrates the core AI/ML components of a **robo-advisory platform** (modeled on Scripbox), built for the Business AI/ML Case Study assignment.

**Flow demonstrated:** `Business Data → AI/ML System → Prediction/Recommendation → Business Action → Business Outcome`

| Component | AI/ML Technique | Business Purpose |
|---|---|---|
| Risk Profiling | Classification (Random Forest) | Categorize investor as Conservative / Moderate / Aggressive |
| Fund Ranking | Multi-factor weighted scoring | Rank mutual funds objectively across categories |
| SIP Projection | Compound growth simulation | Forecast wealth outcomes for the client |
| Recommendation Engine | Rule + ML combination | Match funds to a client's risk profile automatically |

> Data used here is **synthetic** (generated for demonstration), since real Scripbox customer/fund data isn't public. This is explicitly allowed by the assignment — you're demonstrating *how* the AI/ML solution would work, not shipping a production model.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)


## 0. Data Source Configuration

By default this notebook **generates synthetic data** so it runs standalone with zero setup. To run it on your own real data instead:

1. In the cell below, set `USE_SYNTHETIC_CUSTOMER_DATA = False` and/or `USE_SYNTHETIC_FUND_DATA = False`.
2. Run that cell — in Colab, it will prompt you to upload a CSV file.
3. Your CSV just needs these columns (any extra columns are ignored, column names are matched case-insensitively):

**Customer CSV** — `age, monthly_income, dependents, investment_horizon_years, monthly_surplus, existing_investments, market_drop_reaction`
(`risk_profile` is optional — if it's missing, one is auto-generated from the rule-based formula below so the classifier can still be trained.)

**Fund CSV** — `fund_name, category, return_3yr, return_5yr, expense_ratio, std_dev, sharpe_ratio`
(`return_1yr` and `fund_rating` are optional and will default if missing.)


In [ ]:
USE_SYNTHETIC_CUSTOMER_DATA = True   # set to False to upload your own customer data
USE_SYNTHETIC_FUND_DATA = True       # set to False to upload your own fund data

def _upload_csv(label):
    """Prompts a CSV upload in Colab; falls back to a manual path prompt elsewhere."""
    try:
        from google.colab import files
        print(f"Upload your {label} CSV file...")
        uploaded = files.upload()
        fname = list(uploaded.keys())[0]
        return pd.read_csv(fname)
    except ImportError:
        path = input(f"Enter path to your {label} CSV file: ").strip()
        return pd.read_csv(path)

def _standardize_columns(data):
    data = data.copy()
    data.columns = [c.strip().lower().replace(" ", "_") for c in data.columns]
    return data

def label_risk(s):
    if s < 0.4:
        return "Conservative"
    elif s < 0.65:
        return "Moderate"
    else:
        return "Aggressive"


## 1. Synthetic Customer Data

Represents the **Business Data** a robo-advisory platform would actually collect at onboarding:
- Demographic & financial info (age, income, dependents, existing investments)
- Behavioural signals (monthly investable surplus, reaction to past market drops)
- Goal info (investment horizon)

A rule-based risk label (with noise) simulates how a real labelled dataset (from historical client outcomes / advisor assessments) would look.


In [ ]:
CUSTOMER_REQUIRED_COLS = ["age", "monthly_income", "dependents", "investment_horizon_years",
                           "monthly_surplus", "existing_investments", "market_drop_reaction"]

if USE_SYNTHETIC_CUSTOMER_DATA:
    n = 2000
    age = np.random.randint(21, 60, n)
    monthly_income = np.random.randint(25000, 300000, n)
    dependents = np.random.randint(0, 4, n)
    investment_horizon_years = np.random.randint(1, 25, n)
    monthly_surplus = (monthly_income * np.random.uniform(0.05, 0.4, n)).astype(int)
    existing_investments = (monthly_income * np.random.uniform(0, 40, n)).astype(int)
    market_drop_reaction = np.random.randint(1, 6, n)  # 1=panics & sells, 5=stays invested/buys more

    df = pd.DataFrame({
        "age": age,
        "monthly_income": monthly_income,
        "dependents": dependents,
        "investment_horizon_years": investment_horizon_years,
        "monthly_surplus": monthly_surplus,
        "existing_investments": existing_investments,
        "market_drop_reaction": market_drop_reaction,
    })
else:
    df = _standardize_columns(_upload_csv("customer"))
    missing = [c for c in CUSTOMER_REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Your customer CSV is missing required columns: {missing}")

# Rule-based risk score — used as ground truth for synthetic data, and to
# auto-generate risk_profile if your own data doesn't already have it.
risk_score = (
    (df.investment_horizon_years / 25) * 0.35
    + (df.market_drop_reaction / 5) * 0.35
    + (1 - df.dependents / 3) * 0.15
    + (df.monthly_surplus / df.monthly_income).clip(0, 1) * 0.15
)
if USE_SYNTHETIC_CUSTOMER_DATA:
    risk_score += np.random.normal(0, 0.05, n)  # noise, only for synthetic data

if "risk_profile" not in df.columns:
    df["risk_profile"] = [label_risk(s) for s in risk_score]
    if not USE_SYNTHETIC_CUSTOMER_DATA:
        print("Note: 'risk_profile' not found in your data — auto-generated using the rule-based "
              "formula above so the model can still be trained. Replace with real historical "
              "labels if you have them for a more accurate model.")

df.head()


## 2. Risk Profiling Model (Classification)

This is the **AI/ML System** step: a supervised classification model learns to predict a client's risk profile from onboarding data — replacing a manual questionnaire-scoring process with a data-driven, consistent one.

`Random Forest` is used here because it handles mixed numeric features well, is robust to noise, and gives interpretable feature importances — useful for explaining decisions to compliance/clients (a Responsible AI consideration).


In [ ]:
features = ["age", "monthly_income", "dependents", "investment_horizon_years",
            "monthly_surplus", "existing_investments", "market_drop_reaction"]

X = df[features]
le = LabelEncoder()
y = le.fit_transform(df["risk_profile"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred, target_names=le.classes_))


In [ ]:
importances = pd.Series(clf.feature_importances_, index=features).sort_values()
importances.plot(kind="barh", title="What drives the risk-profile prediction?")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()


## 3. Fund Ranking Model (Multi-Factor Scoring)

Simulates the **fund-ranking engine** — instead of clients (or advisors) manually comparing hundreds of mutual funds, the platform scores and ranks them on a weighted composite of performance, risk, cost, and quality factors. This is the kind of transparent, rules-plus-data scoring real robo-advisors use (a lighter-weight version of Scripbox's ~25-factor ranking approach).


In [ ]:
FUND_REQUIRED_COLS = ["fund_name", "category", "return_3yr", "return_5yr",
                       "expense_ratio", "std_dev", "sharpe_ratio"]

if USE_SYNTHETIC_FUND_DATA:
    fund_categories = ["Large Cap", "Mid Cap", "Small Cap", "Debt", "Hybrid"]
    n_funds = 60

    funds = pd.DataFrame({
        "fund_name": [f"Fund_{i+1:02d}" for i in range(n_funds)],
        "category": np.random.choice(fund_categories, n_funds),
        "return_1yr": np.round(np.random.normal(12, 6, n_funds), 2),
        "return_3yr": np.round(np.random.normal(14, 5, n_funds), 2),
        "return_5yr": np.round(np.random.normal(13, 4, n_funds), 2),
        "expense_ratio": np.round(np.random.uniform(0.2, 2.2, n_funds), 2),
        "std_dev": np.round(np.random.uniform(4, 22, n_funds), 2),
        "sharpe_ratio": np.round(np.random.uniform(0.3, 1.8, n_funds), 2),
        "fund_rating": np.random.randint(1, 6, n_funds),
    })
else:
    funds = _standardize_columns(_upload_csv("fund"))
    missing = [c for c in FUND_REQUIRED_COLS if c not in funds.columns]
    if missing:
        raise ValueError(f"Your fund CSV is missing required columns: {missing}")
    if "fund_rating" not in funds.columns:
        funds["fund_rating"] = 3
        print("Note: 'fund_rating' not found — defaulted to 3 (neutral) for all funds.")
    if "return_1yr" not in funds.columns:
        funds["return_1yr"] = funds["return_3yr"]
        print("Note: 'return_1yr' not found — defaulted to return_3yr value.")

def normalize(s, higher_is_better=True):
    s_norm = (s - s.min()) / (s.max() - s.min() + 1e-9)
    return s_norm if higher_is_better else 1 - s_norm

weights = {
    "return_3yr": 0.25,
    "return_5yr": 0.20,
    "sharpe_ratio": 0.20,
    "fund_rating": 0.15,
    "expense_ratio": 0.10,   # lower is better
    "std_dev": 0.10,         # lower is better
}

funds["composite_score"] = (
    normalize(funds.return_3yr) * weights["return_3yr"]
    + normalize(funds.return_5yr) * weights["return_5yr"]
    + normalize(funds.sharpe_ratio) * weights["sharpe_ratio"]
    + normalize(funds.fund_rating) * weights["fund_rating"]
    + normalize(funds.expense_ratio, higher_is_better=False) * weights["expense_ratio"]
    + normalize(funds.std_dev, higher_is_better=False) * weights["std_dev"]
)

funds_ranked = funds.sort_values("composite_score", ascending=False).reset_index(drop=True)
funds_ranked.head(10)


## 4. SIP Projection Engine

Turns the recommendation into a **Business Outcome** the client actually cares about — projected wealth over time. Expected return assumptions are tied to the risk profile, mirroring how a real robo-advisor sets realistic expectations per category.


In [ ]:
expected_annual_return = {
    "Conservative": 0.08,
    "Moderate": 0.11,
    "Aggressive": 0.14,
}

def project_sip(monthly_sip, years, annual_return):
    r = annual_return / 12
    n_months = years * 12
    future_value = monthly_sip * (((1 + r) ** n_months - 1) / r) * (1 + r)
    return future_value

years_range = np.arange(1, 21)

plt.figure()
for profile, rate in expected_annual_return.items():
    values = [project_sip(10000, y, rate) for y in years_range]
    plt.plot(years_range, values, label=profile)

plt.title("SIP Growth Projection — ₹10,000/month by Risk Profile")
plt.xlabel("Years")
plt.ylabel("Projected Value (₹)")
plt.legend()
plt.tight_layout()
plt.show()


## 5. End-to-End Recommendation — Sample Client

Ties every component together for one sample client: profile → risk prediction → matched fund shortlist → wealth projection. This is what "How the AI Solution Works" (Slide 6) should walk through as a flow diagram.


In [ ]:
sample_client = pd.DataFrame([{
    "age": 29,
    "monthly_income": 90000,
    "dependents": 0,
    "investment_horizon_years": 15,
    "monthly_surplus": 25000,
    "existing_investments": 150000,
    "market_drop_reaction": 4,
}])

predicted_class = clf.predict(sample_client[features])[0]
predicted_profile = le.inverse_transform([predicted_class])[0]
print("Predicted risk profile:", predicted_profile)

# Business Action: match top funds for that risk profile
category_map = {
    "Conservative": ["Debt", "Hybrid"],
    "Moderate": ["Hybrid", "Large Cap"],
    "Aggressive": ["Small Cap", "Mid Cap", "Large Cap"],
}
recommended = funds_ranked[funds_ranked.category.isin(category_map[predicted_profile])].head(5)
print("\nTop recommended funds:")
display(recommended[["fund_name", "category", "return_3yr", "sharpe_ratio", "composite_score"]])

# Business Outcome: projected wealth
fv = project_sip(sample_client.monthly_surplus.iloc[0], sample_client.investment_horizon_years.iloc[0],
                  expected_annual_return[predicted_profile])
print(f"\nProjected value after {sample_client.investment_horizon_years.iloc[0]} years "
      f"of ₹{sample_client.monthly_surplus.iloc[0]:,}/month SIP: ₹{fv:,.0f}")


## 6. Responsible AI Notes (for Slide 8)

- **Bias**: risk labels here are rule-generated; a real system must audit for bias across age/income groups so profiling doesn't systematically under- or over-classify certain demographics.
- **Transparency**: feature-importance and composite-score weights are shown to the client/advisor rather than hidden — supports explainability requirements for financial advice.
- **Human oversight**: SEBI-registered robo-advisors in India require a human-reviewable audit trail; the model should support, not replace, advisor sign-off on recommendations.
- **Data privacy**: financial and behavioural data collected here is sensitive — needs encryption, access controls, and consent management.
- **Model risk**: return assumptions and fund scores are projections, not guarantees — mis-set expectations are a real business/regulatory risk.

## 7. Limitations of This Demo
- All customer and fund data is **synthetic**, generated for illustration only.
- Not a real recommendation engine — do not use outputs as actual financial advice.
- A production version would need SEBI/regulatory compliance review, real fund-house data feeds, and much larger, audited training data.
